# 数据集

In [1]:
poems = [
    "床前明月光，疑是地上霜。举头望明月，低头思故乡。",
    "白日依山尽，黄河入海流。欲穷千里目，更上一层楼。",
    "春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少。",
    "千山鸟飞绝，万径人踪灭。孤舟蓑笠翁，独钓寒江雪。",
    "离离原上草，一岁一枯荣。野火烧不尽，春风吹又生。",
    "向晚意不适，驱车登古原。夕阳无限好，只是近黄昏。",
    "空山不见人，但闻人语响。返景入深林，复照青苔上。",
    "月黑雁飞高，单于夜遁逃。欲将轻骑逐，大雪满弓刀。",
    "红豆生南国，春来发几枝。愿君多采撷，此物最相思。",
    "好雨知时节，当春乃发生。随风潜入夜，润物细无声。",
    "移舟泊烟渚，日暮客愁新。野旷天低树，江清月近人。",
    "功盖三分国，名成八阵图。江流石不转，遗恨失吞吴。",
    "清瑟怨遥夜，绕弦风雨哀。孤灯闻楚角，残月下章台。",
    "垂钓坐磐石，水清心亦闲。鱼行潭树下，猿挂岛藤间。",
    "林暗草惊风，将军夜引弓。平明寻白羽，没在石棱中。",
    "山中相送罢，日暮掩柴扉。春草年年绿，王孙归不归。",
    "打起黄莺儿，莫教枝上啼。啼时惊妾梦，不得到辽西。",
    "寥落古行宫，宫花寂寞红。白头宫女在，闲坐说玄宗。",
    "葡萄美酒夜光杯，欲饮琵琶马上催。醉卧沙场君莫笑，古来征战几人回。",
    "秦时明月汉时关，万里长征人未还。但使龙城飞将在，不教胡马度阴山。"
]

# 构建词典

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import Counter

In [6]:
def build_vocab(poems):
    all_chars = []
    for poem in poems:
        all_chars.extend(list(poem))
    counter = Counter(all_chars)
    # 添加特殊标记
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    for char, _ in counter.items():
        if char not in vocab:
            vocab[char] = len(vocab)
    return vocab

In [ ]:
vocab = build_vocab(poems)             #word to id
idx_to_char = {idx: char for char, idx in vocab.items()}    # id to ward
vocab_size = len(vocab)                # 词典大小

In [ ]:
print(vocab)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, '床': 4, '前': 5, '明': 6, '月': 7, '光': 8, '，': 9, '疑': 10, '是': 11, '地': 12, '上': 13, '霜': 14, '。': 15, '举': 16, '头': 17, '望': 18, '低': 19, '思': 20, '故': 21, '乡': 22, '白': 23, '日': 24, '依': 25, '山': 26, '尽': 27, '黄': 28, '河': 29, '入': 30, '海': 31, '流': 32, '欲': 33, '穷': 34, '千': 35, '里': 36, '目': 37, '更': 38, '一': 39, '层': 40, '楼': 41, '春': 42, '眠': 43, '不': 44, '觉': 45, '晓': 46, '处': 47, '闻': 48, '啼': 49, '鸟': 50, '夜': 51, '来': 52, '风': 53, '雨': 54, '声': 55, '花': 56, '落': 57, '知': 58, '多': 59, '少': 60, '飞': 61, '绝': 62, '万': 63, '径': 64, '人': 65, '踪': 66, '灭': 67, '孤': 68, '舟': 69, '蓑': 70, '笠': 71, '翁': 72, '独': 73, '钓': 74, '寒': 75, '江': 76, '雪': 77, '离': 78, '原': 79, '草': 80, '岁': 81, '枯': 82, '荣': 83, '野': 84, '火': 85, '烧': 86, '吹': 87, '又': 88, '生': 89, '向': 90, '晚': 91, '意': 92, '适': 93, '驱': 94, '车': 95, '登': 96, '古': 97, '夕': 98, '阳': 99, '无': 100, '限': 101, '好': 102, '只': 103, '近': 104, '昏': 105, '空': 106, '见': 107, '但': 108, '语'

# 数据集与批次集


In [9]:
# ---------------------- 超参数 ----------------------
EMBEDDING_DIM = 256             # 词向量长度
HIDDEN_DIM    = 512             # 词袋大小
NUM_LAYERS    = 2               # 隐藏层
BATCH_SIZE    = 64              # 递归网络的层数
EPOCHS        = 80              # 批次大小
LEARNING_RATE = 0.001           # 轮数
TEACHER_FORCING_RATIO = 0.5     # 学习率
MAX_LEN       = 30              # 生产的最大长度

# token转换为tensor

In [ ]:
def poem_to_tensor(poem):
    """将诗句转换为tensor,添加SOS和EOS"""
    indices = [vocab.get(ch, vocab['<UNK>']) for ch in poem]                                # 转换为token
    return torch.tensor([vocab['<SOS>']] + indices + [vocab['<EOS>']], dtype=torch.long)    # 转换为tensor张量 

# 对齐

In [12]:
def pad_sequence(seq, max_len, pad_idx):
    """填充序列到max_len"""
    if len(seq) >= max_len:
        return seq[:max_len]   # 截断
    else:
        return torch.cat(
            [
                seq, # 源序列
                torch.full((max_len - len(seq),), pad_idx, dtype=torch.long)   # 补齐为padding的序列
            ]
        )  # 补齐

# 生成源序列

In [14]:
poem_tensors = [poem_to_tensor(poem) for poem in poems]

max_src_len = max(len(t) for t in poem_tensors)
max_tgt_len = max_src_len

In [16]:
print(max_src_len)

34


In [ ]:
# ---------------------- 数据加载器 ----------------------
def get_batches(poem_tensors, batch_size, max_src_len, max_tgt_len, pad_idx):
    # 将每个诗句填充到相同长度
    src_seqs = []
    tgt_seqs = []
    for t in poem_tensors:
        src_padded = pad_sequence(t, max_src_len, pad_idx)
        tgt_padded = pad_sequence(t, max_tgt_len, pad_idx)
        src_seqs.append(src_padded)
        tgt_seqs.append(tgt_padded)
    data = list(zip(src_seqs, tgt_seqs))
    random.shuffle(data)
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        src_batch = torch.stack([item[0] for item in batch])    #原序列 包含<SOS>,  不包含<EOS>
        tgt_batch = torch.stack([item[1] for item in batch])    #目标序列 包含<EOS>,不包含<SOS>
        yield src_batch, tgt_batch

In [17]:
dataloader = get_batches(poem_tensors,BATCH_SIZE,max_src_len,max_tgt_len,vocab["<PAD>"])

for src, tgt in dataloader:
    print(src.shape)
    break

torch.Size([20, 34])


# 模型

In [18]:
class EncoderGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super(EncoderGRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab['<PAD>'])
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        
    def forward(self, x):
        embedded = self.embedding(x)          # (batch, seq_len, embed_dim)
        outputs, hidden = self.gru(embedded)  # hidden: (num_layers, batch, hidden_dim)
        return outputs, hidden

In [19]:
class DecoderGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super(DecoderGRU, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab['<PAD>'])
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden):
        x = x.unsqueeze(1)                    # (batch, 1) -> (batch, 1) 实际上输入是(batch,1)
        embedded = self.embedding(x)          # (batch, 1, embed_dim)
        output, hidden = self.gru(embedded, hidden)   # output: (batch, 1, hidden_dim)
        prediction = self.fc_out(output.squeeze(1))   # (batch, vocab_size)
        return prediction, hidden

In [21]:
class Seq2SeqGRU(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2SeqGRU, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[0]
        max_len = target.shape[1]
        output_dim = self.decoder.fc_out.out_features
        
        outputs = torch.zeros(batch_size, max_len, output_dim).to(source.device)
        
        # 编码
        _, hidden = self.encoder(source)
        
        # 解码第一个输入是SOS
        decoder_input = target[:, 0]   # SOS token
        
        for t in range(1, max_len):
            prediction, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t, :] = prediction
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            decoder_input = target[:, t] if teacher_force else top1
        
        return outputs
    
    def generate(self, start_chars, max_len=MAX_LEN):
        """根据给定的起始字符生成诗句"""
        self.eval()
        model_device = next(self.encoder.parameters()).device
        with torch.no_grad():
            # 将起始字符转换为索引tensor
            indices = [vocab.get(ch, vocab['<UNK>']) for ch in start_chars]
            src_tensor = torch.tensor([vocab['<SOS>']] + indices, dtype=torch.long).unsqueeze(0).to(model_device)
            # 编码
            _, hidden = self.encoder(src_tensor)
            # 解码
            decoder_input = torch.tensor([vocab['<SOS>']], device=src_tensor.device)
            generated = list(start_chars)
            for _ in range(max_len):
                
                prediction, hidden = self.decoder(decoder_input, hidden)
                top1 = prediction.argmax(1).item()
                if top1 == vocab['<EOS>']:
                    break
                generated.append(idx_to_char[top1])
                decoder_input = torch.tensor([top1], device=src_tensor.device)
        return ''.join(generated)

# 训练

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"使用设备: {device}")

使用设备: cpu


In [25]:
encoder = EncoderGRU(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
decoder = DecoderGRU(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS)
model = Seq2SeqGRU(encoder, decoder).to(device)

In [26]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab['<PAD>'])
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [27]:
pad_idx = vocab['<PAD>']
sos_idx = vocab['<SOS>']
eos_idx = vocab['<EOS>']

In [28]:
print("开始训练...")
for epoch in range(EPOCHS):
    total_loss = 0
    batches = get_batches(poem_tensors, BATCH_SIZE, max_src_len, max_tgt_len, pad_idx)
    for src, tgt in batches:
        src, tgt = src.to(device), tgt.to(device)

        # 梯度清零
        optimizer.zero_grad()

        #推理
        output = model(src, tgt, teacher_forcing_ratio=TEACHER_FORCING_RATIO)
        # 输出形状: (batch, max_len, vocab_size)，忽略第0个位置的SOS
        output = output[:, 1:, :].reshape(-1, vocab_size)
        target = tgt[:, 1:].reshape(-1)
        loss = criterion(output, target)

        # 自动求导
        loss.backward()

        #裁剪（防止梯度爆炸） 权重系数超过1 都比为1
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(poems):.4f}")

开始训练...
Epoch 10/80, Loss: 0.2344
Epoch 20/80, Loss: 0.2047
Epoch 30/80, Loss: 0.1847
Epoch 40/80, Loss: 0.1503
Epoch 50/80, Loss: 0.1327
Epoch 60/80, Loss: 0.0812
Epoch 70/80, Loss: 0.0391
Epoch 80/80, Loss: 0.0171


# 生成唐诗

In [34]:
print("\n生成唐诗示例:")
model.eval()
test_starts = ["床前", "白日", "春眠", "千山", "向晚", "月黑"]
for start in test_starts:
    generated = model.generate(start, max_len=20)
    print(f"起始: {start} -> {generated}")

# 保存模型（可选）
torch.save(model.state_dict(), "/Users/logicye/Code/ai_learning/notebooks/06_多模态/seq2seq_tang_poem.pth")


生成唐诗示例:


AssertionError: Torch not compiled with CUDA enabled